# SecBERT Fine-Tuning for Cybersecurity NER

```
jackaduma/SecBERT  (pre-trained with MLM on cybersecurity corpora by its original authors)
        ↓
Fine-tune for Cybersecurity NER (Token Classification)
        ↓
Evaluate (Precision, Recall, F1)
        ↓
Demo
```

The base model, [`jackaduma/SecBERT`](https://huggingface.co/jackaduma/SecBERT), was pre-trained with Masked Language Modeling on security-domain corpora (APTnotes, Stucco-Data, CASIE, SemEval-2018 Task 8) by its original authors. This notebook fine-tunes it for cybersecurity Named Entity Recognition on the CyberNER dataset (31-label BIO schema, 15 entity types) and evaluates it with entity-level Precision, Recall, and F1.

Requires a GPU runtime: **Runtime → Change runtime type → T4 GPU**.

In [1]:
%pip install -q -U transformers datasets evaluate seqeval accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.3 MB/s eta 0:00:00


## 1. Load dataset files from Google Drive

Reads `cyberner_clean.csv` and `ner_cyber_labels.json` from `MyDrive/datasets/ner/`.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/datasets/ner"
OUTPUT_DIR = "/content/drive/MyDrive/models"

!mkdir -p "{OUTPUT_DIR}"
!cp "{DATA_DIR}/cyberner_clean.csv" "{DATA_DIR}/ner_cyber_labels.json" .

Mounted at /content/drive


## 2. Prepare the CyberNER dataset

Raw tags are mapped to the 31-label cyber schema; unmapped tags become `O`. Tokens are grouped into sentences by `Sentence_ID`.

In [3]:
import json
import pandas as pd

with open("ner_cyber_labels.json", encoding="utf-8") as f:
    schema = json.load(f)

label_list = schema["label_list"]
tag_mapping = schema["tag_mapping"]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

df = pd.read_csv("cyberner_clean.csv")
df["Word"] = df["Word"].fillna("#")
df["Tag"] = df["Tag"].fillna("O").map(lambda t: tag_mapping.get(t, "O"))

grouped = df.groupby("Sentence_ID").agg({"Word": list, "Tag": list}).reset_index()
grouped["ner_tags"] = grouped["Tag"].apply(lambda tags: [label2id.get(t, label2id["O"]) for t in tags])
grouped = grouped.rename(columns={"Word": "tokens"})

print(f"Labels: {len(label_list)}  |  Sentences: {len(grouped)}")

Labels: 31  |  Sentences: 10042


## 3. Train / validation / test splits

80/20 train/test, then 80/20 of train for validation (seed 42). `scripts/evaluate_ner.py` re-derives the same test split locally.

In [4]:
from datasets import Dataset

full_ds = Dataset.from_pandas(grouped[["tokens", "ner_tags"]])
split1 = full_ds.train_test_split(test_size=0.2, seed=42)
split2 = split1["train"].train_test_split(test_size=0.2, seed=42)
train_ds, val_ds, test_ds = split2["train"], split2["test"], split1["test"]

print(f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")

train=6426  val=1607  test=2009


## 4. Tokenization and label alignment

Only the first sub-token of each word carries the label; remaining sub-tokens and special tokens are set to `-100` and ignored by the loss and metrics.

In [5]:
from transformers import AutoTokenizer

MODEL_NAME = "jackaduma/SecBERT"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=512,
        is_split_into_words=True,
    )
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_train = train_ds.map(tokenize_and_align_labels, batched=True)
tokenized_val = val_ds.map(tokenize_and_align_labels, batched=True)
tokenized_test = test_ds.map(tokenize_and_align_labels, batched=True)

config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/378k [00:00<?, ?B/s]

Map:   0%|          | 0/6426 [00:00<?, ? examples/s]

Map:   0%|          | 0/1607 [00:00<?, ? examples/s]

Map:   0%|          | 0/2009 [00:00<?, ? examples/s]

## 5. Load SecBERT with a token-classification head

The warning that classifier weights are newly initialized is expected — that head is what fine-tuning trains.

In [6]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

model.safetensors:   0%|          | 0.00/336M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: jackaduma/SecBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 6. Evaluation metrics (entity-level Precision, Recall, F1)

In [7]:
import numpy as np
import evaluate

seqeval_metric = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    true_predictions = [
        [id2label[p] for p, l in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for p, l in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    results = seqeval_metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

## 7. Fine-tune

In [8]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

training_args = TrainingArguments(
    output_dir="secbert_ner",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.255134,0.447280,0.424474,0.435579,0.929501
2,0.373160,0.225334,0.554153,0.484810,0.517167,0.939357
3,0.194389,0.216947,0.531326,0.558530,0.544588,0.940712
4,0.124451,0.223147,0.523238,0.593159,0.556009,0.940432
5,0.085660,0.231187,0.549961,0.605694,0.576484,0.943506
6,0.085660,0.243633,0.554277,0.613979,0.582603,0.943979
7,0.055877,0.258225,0.550808,0.629913,0.587711,0.943711
8,0.041009,0.268600,0.562833,0.628001,0.593634,0.944656
9,0.032246,0.268154,0.572892,0.617803,0.594501,0.946107
10,0.027793,0.273035,0.570593,0.621627,0.595018,0.945946


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4020, training_loss=0.11634495798035048, metrics={'train_runtime': 431.2529, 'train_samples_per_second': 149.008, 'train_steps_per_second': 9.322, 'total_flos': 2747016565708032.0, 'train_loss': 0.11634495798035048, 'epoch': 10.0})

## 8. Evaluate on the held-out test set

In [9]:
from seqeval.metrics import classification_report
from seqeval.scheme import IOB2

test_output = trainer.predict(tokenized_test)

print("Test-set entity-level metrics:")
for key in ("test_precision", "test_recall", "test_f1", "test_accuracy"):
    print(f"  {key.replace('test_', '').capitalize():<10} {test_output.metrics[key]:.4f}")

predictions = np.argmax(test_output.predictions, axis=-1)
labels = test_output.label_ids
true_predictions = [
    [id2label[p] for p, l in zip(pred, lab) if l != -100]
    for pred, lab in zip(predictions, labels)
]
true_labels = [
    [id2label[l] for p, l in zip(pred, lab) if l != -100]
    for pred, lab in zip(predictions, labels)
]

print(classification_report(true_labels, true_predictions, mode="strict", scheme=IOB2, zero_division=0))

Test-set entity-level metrics:
  Precision  0.5670
  Recall     0.6290
  F1         0.5964
  Accuracy   0.9420
                precision    recall  f1-score   support

           APT       0.71      0.72      0.71       799
      CAMPAIGN       0.31      0.12      0.18        40
       EXPLOIT       0.81      0.86      0.84       247
          FILE       0.67      0.69      0.68       676
          HASH       0.74      0.82      0.77        92
     INDICATOR       0.71      0.71      0.71       272
INFRASTRUCTURE       0.43      0.50      0.46       151
            IP       0.73      0.80      0.76        65
       MALWARE       0.63      0.64      0.64      1140
        METHOD       0.32      0.26      0.29       370
      SOFTWARE       0.61      0.63      0.62       302
  THREAT_ACTOR       0.47      0.39      0.43       170
          TOOL       0.59      0.57      0.58      1407
           URL       0.65      0.68      0.67        19
 VULNERABILITY       0.77      0.68      0.72   

## 9. Save the model to Google Drive (`MyDrive/models/`)

In [10]:
trainer.save_model("secbert_ner_final")
tokenizer.save_pretrained("secbert_ner_final")

!zip -r -q secbert_ner_final.zip secbert_ner_final
!cp secbert_ner_final.zip "{OUTPUT_DIR}/"

print(f"Saved to {OUTPUT_DIR}/secbert_ner_final.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /content/drive/MyDrive/models/secbert_ner_final.zip


## 10. Deploy locally

1. Download `secbert_ner_final.zip` from `MyDrive/models/` and extract it to `models/secbert_ner_final/` in the project root.
2. Reproduce the evaluation: `python scripts/evaluate_ner.py`
3. Run the demo: `python backend/ner_api.py`, then `cd frontend && npm start`.